In [64]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image, ImageDraw

tf.random.set_seed(42)
np.random.seed(42)

IMG_SIZE = 640
GRID_SIZE = 20
BATCH_SIZE = 8


In [65]:
base = Path("/kaggle/input")

dataset_root = None

for yaml_file in base.rglob("data.yaml"):
    root = yaml_file.parent

    correct_structure = all(
        (root / split / "images").exists()
        and (root / split / "labels").exists()
        for split in ["train", "valid", "test"]
    )

    if correct_structure:
        dataset_root = root
        break


if dataset_root is None:
    raise FileNotFoundError("Person dataset could not be found.")


print("Dataset:", dataset_root)

Dataset: /kaggle/input/datasets/k240657rahoulkumar/person-yolo


In [66]:
def path_create(split):

    image_path = str(dataset_root / split / "images")
    label_path = str(dataset_root / split / "labels")

    return image_path, label_path


train_P_I, train_P_L = path_create("train")
valid_P_I, valid_P_L = path_create("valid")
test_P_I, test_P_L = path_create("test")

print(train_P_I)
print(valid_P_I)
print(test_P_I)

/kaggle/input/datasets/k240657rahoulkumar/person-yolo/train/images
/kaggle/input/datasets/k240657rahoulkumar/person-yolo/valid/images
/kaggle/input/datasets/k240657rahoulkumar/person-yolo/test/images


In [67]:
def create_pairs(image_folder, label_folder):

    pairs = []

    for image_name in sorted(os.listdir(image_folder)):

        if not image_name.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):
            continue

        label_name = os.path.splitext(image_name)[0] + ".txt"

        image_path = os.path.join(
            image_folder,
            image_name
        )

        label_path = os.path.join(
            label_folder,
            label_name
        )

        if os.path.exists(label_path):
            pairs.append(
                (image_path, label_path)
            )

    return pairs

In [68]:
train_pairs = create_pairs(
    train_P_I,
    train_P_L
)

valid_pairs = create_pairs(
    valid_P_I,
    valid_P_L
)

test_pairs = create_pairs(
    test_P_I,
    test_P_L
)


print("Train:", len(train_pairs))
print("Validation:", len(valid_pairs))
print("Test:", len(test_pairs))

Train: 3743
Validation: 1075
Test: 539


In [69]:
def create_target(label_path):

    label_path = label_path.numpy().decode("utf-8")

    target = np.zeros(
        (GRID_SIZE, GRID_SIZE, 5),
        dtype=np.float32
    )

    with open(label_path, "r") as file:

        for line in file:

            values = line.strip().split()

            if len(values) != 5:
                continue

            class_id = int(float(values[0]))

            x_center = float(values[1])
            y_center = float(values[2])
            width = float(values[3])
            height = float(values[4])

            # We only want the person class
            if class_id != 0:
                continue

            x_center = np.clip(
                x_center,
                0.0,
                0.999999
            )

            y_center = np.clip(
                y_center,
                0.0,
                0.999999
            )

            grid_x = int(
                x_center * GRID_SIZE
            )

            grid_y = int(
                y_center * GRID_SIZE
            )

            x_inside_cell = (
                x_center * GRID_SIZE
                - grid_x
            )

            y_inside_cell = (
                y_center * GRID_SIZE
                - grid_y
            )

            new_area = width * height

            old_area = (
                target[grid_y, grid_x, 3]
                *
                target[grid_y, grid_x, 4]
            )

            # If two people have centers in the
            # same cell, keep the larger box.
            if (
                target[grid_y, grid_x, 0] == 0
                or new_area > old_area
            ):

                target[
                    grid_y,
                    grid_x
                ] = [
                    1.0,
                    x_inside_cell,
                    y_inside_cell,
                    width,
                    height
                ]

    return target
    

In [70]:
def load_sample(image_path, label_path):

    image_data = tf.io.read_file(image_path)

    image = tf.image.decode_jpeg(
        image_data,
        channels=3
    )

    image = tf.image.convert_image_dtype(
        image,
        tf.float32
    )

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    target = tf.py_function(
        create_target,
        [label_path],
        Tout=tf.float32
    )

    target.set_shape(
        [GRID_SIZE, GRID_SIZE, 5]
    )

    return image, target

In [71]:
def create_tf_dataset(
    pairs,
    batch_size=8,
    shuffle=False
):

    image_paths = [
        pair[0]
        for pair in pairs
    ]

    label_paths = [
        pair[1]
        for pair in pairs
    ]

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, label_paths)
    )

    if shuffle:

        dataset = dataset.shuffle(
            len(pairs)
        )

    dataset = dataset.map(
        load_sample,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(
        batch_size
    )

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset

In [72]:
train_dataset = create_tf_dataset(
    train_pairs,
    BATCH_SIZE,
    shuffle=True
)

valid_dataset = create_tf_dataset(
    valid_pairs,
    BATCH_SIZE
)

test_dataset = create_tf_dataset(
    test_pairs,
    BATCH_SIZE
)

In [75]:
from tensorflow.keras import layers

model = tf.keras.Sequential(
    [
        layers.Input(shape=(640, 640, 3)),

        # 640 -> 320
        layers.Conv2D(16, 3, padding="same"),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.MaxPooling2D(2),

        # 320 -> 160
        layers.Conv2D(32, 3, padding="same"),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.MaxPooling2D(2),

        # 160 -> 80
        layers.Conv2D(64, 3, padding="same"),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.MaxPooling2D(2),

        # 80 -> 40
        layers.Conv2D(128, 3, padding="same"),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.MaxPooling2D(2),

        # 40 -> 20
        layers.Conv2D(256, 3, padding="same"),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.MaxPooling2D(2),

        layers.Conv2D(
            5,
            1,
            activation="sigmoid"
        )
    ]
)

In [80]:
def yolo_loss(y_true,y_pred):

    epsilon = 1e-7

    true_object = y_true[..., 0]
    pred_object = y_pred[..., 0]

    pred_object = tf.clip_by_value(
        pred_object,
        epsilon,
        1.0 - epsilon
    )

    object_bce = -(
        true_object
        *
        tf.math.log(pred_object)
        +
        (1.0 - true_object)
        *
        tf.math.log(
            1.0 - pred_object
        )
    )
    
    object_weights = (
        true_object * 5.0
        +
        (1.0 - true_object) * 0.5
    )

    object_loss = tf.reduce_mean(
        object_weights
        *
        object_bce
    )


    true_boxes = y_true[..., 1:5]
    pred_boxes = y_pred[..., 1:5]

    coordinate_error = tf.reduce_sum(
        tf.square(
            true_boxes
            -
            pred_boxes
        ),
        axis=-1
    )

    coordinate_loss = (
        tf.reduce_sum(
            coordinate_error
            *
            true_object
        )
        /
        (
            tf.reduce_sum(
                true_object
            )
            +
            epsilon
        )
    )

    total_loss = (
        object_loss
        +
        5.0 * coordinate_loss
    )

    return total_loss

In [81]:
def cell_precision(
    y_true,
    y_pred
):

    true_obj = (
        y_true[..., 0] > 0.5
    )

    pred_obj = (
        y_pred[..., 0] > 0.5
    )

    tp = tf.reduce_sum(
        tf.cast(
            true_obj & pred_obj,
            tf.float32
        )
    )

    predicted = tf.reduce_sum(
        tf.cast(
            pred_obj,
            tf.float32
        )
    )

    return (
        tp
        /
        (predicted + 1e-7)
    )

In [82]:
def cell_recall(
    y_true,
    y_pred
):

    true_obj = (
        y_true[..., 0] > 0.5
    )

    pred_obj = (
        y_pred[..., 0] > 0.5
    )

    tp = tf.reduce_sum(
        tf.cast(
            true_obj & pred_obj,
            tf.float32
        )
    )

    actual = tf.reduce_sum(
        tf.cast(
            true_obj,
            tf.float32
        )
    )

    return (
        tp
        /
        (actual + 1e-7)
    )

In [83]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=yolo_loss,
    metrics=[
        cell_precision,
        cell_recall
    ]
)

In [84]:
checkpoint_path = (
    "/kaggle/working/"
    "best_human_counter.weights.h5"
)


callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
history=model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30
 30/468 ━━━━━━━━━━━━━━━━━━━━ 26:13 4s/step - cell_precision: 0.0072 - cell_recall: 0.0570 - loss: 3.5586

In [ ]:
test_results = model.evaluate(
    test_dataset,
    verbose=1
)

print(
    "Test results:",
    test_results
)

In [ ]:
model.save(
    "/kaggle/working/"
    "human_counter_yolo.keras"
)

print(
    "Model saved successfully."
)